<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%20016/Modelos_de_Ensamble_y_Boosting_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Modelos de Ensamble y Boosting Models
### Coderhouse - Data Science
Profe Jorge Ruiz

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import accuracy_score



In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/Jorge-Ruiz-Troccoli/Data-Science-II/refs/heads/main/Clase%20016/winequality-red.csv')
data

In [ ]:
data.quality.unique()

In [ ]:
data.quality.value_counts(normalize=True)*100

Vamos a hacer un problema de clasificacion. Asi que lo que vamos a predecir es
si el vino es de buena calidad o no. Tomaremos el 6 como umbral para decidir si
es bueno o no.

In [ ]:
data.loc[data['quality'] < 6, 'quality'] = 0 #baja calidad
data.loc[data['quality'] >= 6, 'quality'] = 1 #alta calidad

In [ ]:
#Veamos que tenemos
data

In [ ]:
X = data.drop("quality", axis=1) #Elimino de mi dataset la variable a predecir
y = data.quality #Defino el Target

In [ ]:
round(y.value_counts(normalize=True)*100,1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state=123, stratify=y)

# Decision Tree - Clasificación

In [ ]:
clf_tree = DecisionTreeClassifier()

In [ ]:
len(data)

In [ ]:
param_grid = {
    'max_depth': [5, 10, 15],  # Profundidad máxima
    'min_samples_split': [10, 20, 30],  # Número mínimo de muestras requeridas para dividir un nodo interno
     'min_samples_leaf': [5,9]  # Número mínimo de muestras requeridas en cada hoja del árbol
}

clf_tree_cv = HalvingGridSearchCV(clf_tree, param_grid=param_grid, cv=5, min_resources=150, scoring='accuracy', n_jobs=-1, verbose=3)

clf_tree_cv.fit(X_train, y_train)

In [ ]:
y_train_pred = clf_tree_cv.predict(X_train) #Prediccion en Train
y_test_pred = clf_tree_cv.predict(X_test) #Prediccion en Test
# Calcular la precisión
accuracy_tree = accuracy_score(y_test, y_test_pred)
print('DecisionTree Model accuracy score: {0:0.3f}'.format(accuracy_tree))

In [ ]:
# Obtener los scores de todas las combinaciones de parámetros
results = clf_tree_cv.cv_results_
results

# guardemos lo más importante dentro de un dataframe
df_clf_tree_cv = pd.DataFrame(results)

df_clf_tree_cv["mean_test_score_10%"]=accuracy_tree

df_clf_tree_cv = df_clf_tree_cv[['mean_train_score', 'mean_test_score',  "mean_test_score_10%" ,'params']]
df_clf_tree_cv= df_clf_tree_cv.round(3).sort_values("mean_test_score", ascending=False)[:1]
df_clf_tree_cv["model"]="DecisionTree"

df_clf_tree_cv

# Xgboost - Clasificación

In [ ]:
!pip install xgboost

# https://xgboost.readthedocs.io/en/stable/tutorials/model.html
# https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html

In [ ]:
import xgboost as xgb #pip install xgboost

In [ ]:
clf_xgb = xgb.XGBClassifier()

In [ ]:
np.arange(0.1, 0.8,0.2)

In [ ]:
[i/10.0 for i in range(1, 8, 2)]

In [37]:
param_grid = {
    'n_estimators': np.arange(100, 1000, 200),  # Número de árboles en el bosque
    #'max_depth': [5, 10, 15],  # Profundidad máxima de cada árbol
    #'min_samples_split': [10, 20, 30],  # Número mínimo de muestras requeridas para dividir un nodo interno
    #'min_samples_leaf': [5, 9],  # Número mínimo de muestras requeridas en cada hoja del árbol
    'gamma': [i/10.0 for i in range(1, 8, 2)],
    'alpha': np.arange(0.01, 0.08, 0.02),
    'lambda':[i/10.0 for i in range(1,8,2)],
}

clf_xgb_cv = HalvingGridSearchCV(clf_xgb, param_grid=param_grid, cv=5, min_resources=100, scoring='accuracy', n_jobs=-1, verbose=2)

clf_xgb_cv.fit(X_train, y_train)

n_iterations: 3
n_required_iterations: 6
n_possible_iterations: 3
min_resources_: 100
max_resources_: 1439
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 320
n_resources: 100
Fitting 5 folds for each of 320 candidates, totalling 1600 fits
----------
iter: 1
n_candidates: 107
n_resources: 300
Fitting 5 folds for each of 107 candidates, totalling 535 fits
----------
iter: 2
n_candidates: 36
n_resources: 900
Fitting 5 folds for each of 36 candidates, totalling 180 fits


HalvingGridSearchCV(estimator=XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=N...
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...),
                    min_resources=100, n_jobs=-1,
                    param_grid={'alpha': array([0.01, 0.03, 0.05, 0.07]),
                                'gamma': [0.1, 0.3, 0.5, 0.7],
                                'lambda': [0.1, 0.3, 0.5, 0.7],
                                'n_estimators': array([100, 300, 500, 700, 900])},
                    scoring='accuracy', verbose=2)

Regularización (reg_alpha y reg_lambda): Estos hiperparámetros controlan la regularización aplicada a los pesos de los árboles. reg_alpha se refiere a la regularización L1 (LASSO) y reg_lambda se refiere a la regularización L2 (Ridge). Ambos hiperparámetros ayudan a prevenir el sobreajuste.

L2 (lambda) → se usa para reducir overfitting.

L1 (alpha) → para selección de features implícita, independientemente del overfitting. Si hay 200 features y queremos que el modelo "ignore" las irrelevantes llevándolas a peso 0, subimos alpha aunque no haya overfitting.

Overfit → restringimos (bajamos complejidad, subimos regularización)
Underfit → liberamos (subimos complejidad, bajamos regularización)

In [ ]:
y_train_pred = clf_xgb_cv.predict(X_train) #Prediccion en Train
y_test_pred = clf_xgb_cv.predict(X_test) #Prediccion en Test
# Calcular la precisión
accuracy_XGBoost = accuracy_score(y_test, y_test_pred)
print('XGBoost Model accuracy score: {0:0.3f}'.format(accuracy_XGBoost ))

In [ ]:
# Obtener los scores de todas las combinaciones de parámetros
results = clf_xgb_cv.cv_results_
results

# guardemos lo más importante dentro de un dataframe
df_clf_xgb_cv = pd.DataFrame(results)

df_clf_xgb_cv["mean_test_score_10%"]=accuracy_XGBoost

df_clf_xgb_cv = df_clf_xgb_cv[['mean_train_score', 'mean_test_score',  "mean_test_score_10%" ,'params']]
df_clf_xgb_cv= df_clf_xgb_cv.round(3).sort_values("mean_test_score", ascending=False)[:1]
df_clf_xgb_cv["model"]="XGBoost"

df_clf_xgb_cv

# LightGBM - Clasificación

In [ ]:
pip install lightgbm

In [ ]:
import lightgbm as lgb #pip install lightgbm

In [ ]:
clf_LightGBM  = lgb.LGBMClassifier()

In [ ]:
#Definicion de Hyperparámetros

param_grid = {'n_estimators': np.arange(100,400,100),  # Número de árboles en el bosque
    'max_depth': [5, 10, 15],  # Profundidad máxima de cada árbol
    #'min_samples_split': [10, 20, 30],  # Número mínimo de muestras requeridas para dividir un nodo interno
    #'min_samples_leaf': [5,9],  # Número mínimo de muestras requeridas en cada hoja del árbol
    'lambda':[i/10.0 for i in range(1,8,2)],
    'alpha':np.arange(0.1,0.8,0.2)
}

clf_LightGBM_cv = HalvingGridSearchCV(clf_LightGBM, param_grid=param_grid, cv=3, min_resources=100, scoring='accuracy', n_jobs=-1, verbose=2)

clf_LightGBM_cv.fit(X_train, y_train)

In [ ]:
y_train_pred = clf_LightGBM_cv.predict(X_train) #Prediccion en Train
y_test_pred = clf_LightGBM_cv.predict(X_test) #Prediccion en Test
# Calcular la precisión

accuracy_LightGBM = accuracy_score(y_test, y_test_pred)
print('LightGBM Model accuracy score: {0:0.3f}'.format(accuracy_LightGBM))

In [ ]:
# Obtener los scores de todas las combinaciones de parámetros
results = clf_LightGBM_cv.cv_results_
results

# guardemos lo más importante dentro de un dataframe
df_clf_LightGBM_cv = pd.DataFrame(results)

df_clf_LightGBM_cv["mean_test_score_10%"]=accuracy_LightGBM

df_clf_LightGBM_cv = df_clf_LightGBM_cv[['mean_train_score', 'mean_test_score',  "mean_test_score_10%" ,'params']]
df_clf_LightGBM_cv= df_clf_LightGBM_cv.round(3).sort_values("mean_test_score", ascending=False)[:1]
df_clf_LightGBM_cv["model"]="LightGBM"

df_clf_LightGBM_cv

# Gradient Boosting - Clasificación

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier as GBC
gbrt = GBC()

In [ ]:
#Definicion de Hiperparámetros

param_grid = {'n_estimators': np.arange(100,300,100),  # Número de árboles en el bosque
    'max_depth': [5, 15],  # Profundidad máxima de cada árbol
    #'min_samples_split': [10, 20, 30],  # Número mínimo de muestras requeridas para dividir un nodo interno
    #'min_samples_leaf': [5,9],  # Número mínimo de muestras requeridas en cada hoja del árbol
    'ccp_alpha':[0.07,0.08]
}

GBC_cv = HalvingGridSearchCV(gbrt , param_grid=param_grid, cv=3, min_resources=100, scoring='accuracy', n_jobs=-1, verbose=2)

GBC_cv.fit(X_train, y_train)

In [ ]:
y_train_pred = GBC_cv.predict(X_train) #Prediccion en Train
y_test_pred = GBC_cv.predict(X_test) #Prediccion en Test
# Calcular la precisión

accuracy_GBC = accuracy_score(y_test, y_test_pred)
print('GBC Model accuracy score: {0:0.3f}'.format(accuracy_GBC))

In [ ]:
results = GBC_cv.cv_results_
results

# guardemos lo más importante dentro de un dataframe
df_clf_GBC_cv = pd.DataFrame(results)

df_clf_GBC_cv["mean_test_score_10%"]=accuracy_GBC

df_clf_GBC_cv = df_clf_GBC_cv[['mean_train_score', 'mean_test_score',  "mean_test_score_10%" ,'params']]
df_clf_GBC_cv= df_clf_GBC_cv.round(3).sort_values("mean_test_score", ascending=False)[:1]
df_clf_GBC_cv["model"]="GBC"

df_clf_GBC_cv

In [ ]:
df_resultados= pd.concat([df_clf_LightGBM_cv, df_clf_xgb_cv, df_clf_GBC_cv, df_clf_tree_cv], axis=0)
df_resultados

#Importante!!!
Para regresión es lo mismo, solo se debe cambiar a modo regressor. Por ejemplo: xgb.XGBRegressor() y el scoring a seleccionar debe ser compatible, como podría ser 'neg_mean_squared_error'

Probemos una libreria bastante interesante dentro de ML:

In [ ]:
pip install lazypredict

In [ ]:
# Importing LazyRegressor
from lazypredict.Supervised import LazyClassifier
# https://pypi.org/project/lazypredict/

#https://www.geeksforgeeks.org/lazy-predict-library-in-python-for-machine-learning/

In [ ]:
lazy_clf= LazyClassifier(verbose=0,
					ignore_warnings=False,
					custom_metric=None)

# fitting data in LazyClassifier
models, predictions = lazy_clf.fit(X_train, X_test,
							y_train, y_test)
models


Material complementario:

https://towardsdatascience.com/what-when-how-extratrees-classifier-c939f905851c

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html

https://scikit-learn.org/stable/modules/generated/sklearn.semi_supervised.LabelPropagation.html